# Query Decomposition RAG Pipeline (Xumo Manual)

## Concept
This code demonstrates **Query Decomposition in a RAG pipeline**.  
Instead of answering a complex question directly, the system **splits it into smaller sub‑questions**, retrieves context for each, and then combines the answers into a final response.

---

## ⚙️ How the Code Works
1. **Load Knowledge Base**  
   - The Xumo Stream Box manual (`xumo_manual_rag.txt`) is loaded.  
   - Text is split into chunks and embedded using **Hugging Face embeddings**.  
   - Stored in a **FAISS vector database** for fast retrieval.

2. **Initialize LLM**  
   - A **Groq LLM (llama‑3.1‑8b‑instant)** is used to generate answers and perform query decomposition.

3. **State Schema**  
   - Defines a `DecomposeRAGState` object with fields for the question, sub‑questions, retrieved docs, and final answer.

4. **Nodes in the Pipeline**  
   - **Planner Node**: Breaks the complex query into 2–3 sub‑questions.  
   - **Retriever Node**: Fetches relevant manual chunks for each sub‑question.  
   - **Responder Node**: Generates the final answer using all retrieved context.  

5. **LangGraph Flow**  
   - Planner → Retriever → Responder → Final Answer.  
   - Ensures structured decomposition and retrieval before answering.

6. **Gradio Interface**  
   - Provides a simple UI where users can type complex queries.  
   - Displays both the **decomposed sub‑questions** and the **final answer**.

---

## Example Run

**Complex Query:**  
*Is stream support 4K and what are the features?*

**Decomposed RAG Output:**

### 🔍 Sub‑questions
- **Stream support for 4K**  
  - Is the streaming platform capable of handling 4K resolution streams?  
  - Are there any limitations on bitrate, frame rate, or other technical specifications?  

- **Features for 4K streaming**  
  - What features support 4K streaming (HDR, Dolby Vision, Atmos)?  
  - Are there advanced features like dynamic metadata or caption customization?  

- **Hardware and software requirements**  
  - What are the minimum hardware requirements for 4K streaming?  
  - Are there specific software configurations needed?

---

### Final Answer
The manual does not explicitly confirm **4K streaming support**.  
However, it mentions **Dolby Vision HDR** and **Dolby Atmos audio**, both of which are typically associated with 4K content.  
This suggests possible 4K capability, but it is not guaranteed.

Additional features listed in the manual include:
- High‑contrast display mode  
- Voice guidance  
- HDMI control  
- Audio description (AD/SAP)  
- Closed captions & customized caption style  
- Highlight programs  
- Beep on audio description  

---

## Relevance to RAG Workflows
- **Improves retrieval accuracy** by focusing on atomic sub‑questions.  
- **Enhances response quality** by grounding each part of the query.  
- **Useful for technical manuals** like the Xumo Stream Box, where queries often span multiple features.  

---

**Takeaway**: Query decomposition makes RAG pipelines smarter — breaking down complex queries ensures **precise retrieval** and **transparent answers**.


In [8]:
# ---------------------------------
# 0. Setup & Imports
# ---------------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [9]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ---------------------------------
# 1. Load and Embed Documents
# ---------------------------------
manual = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(manual)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [3]:
# ---------------------------------
# 2. Initialize Groq LLM
# ---------------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [4]:
# ---------------------------------
# 3. State Schema
# ---------------------------------
class DecomposeRAGState(BaseModel):
    question: str
    sub_questions: List[str] = []
    retrieved_docs: List[Document] = []
    answer: str = ""

In [5]:
# ---------------------------------
# 4. Nodes
# ---------------------------------

# a. Query Planner: splits input question
def plan_query(state: DecomposeRAGState) -> DecomposeRAGState:
    prompt = f"""
Break the following complex question into 2-3 sub-questions:

Question: {state.question}

Sub-questions:
"""
    result = llm.invoke(prompt)
    sub_questions = [line.strip("- ").strip() for line in result.content.strip().split("\n") if line.strip()]
    return DecomposeRAGState(question=state.question, sub_questions=sub_questions)

# b. Retrieve documents for each sub-question
def retrieve_for_each(state: DecomposeRAGState) -> DecomposeRAGState:
    all_docs = []
    for sub in state.sub_questions:
        docs = retriever.invoke(sub)
        all_docs.extend(docs)
    return DecomposeRAGState(question=state.question, sub_questions=state.sub_questions, retrieved_docs=all_docs)

# c. Generate final answer
def generate_final_answer(state: DecomposeRAGState) -> DecomposeRAGState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question: {state.question}
"""
    answer = llm.invoke(prompt).content
    return DecomposeRAGState(
        question=state.question,
        sub_questions=state.sub_questions,
        retrieved_docs=state.retrieved_docs,
        answer=answer
    )

In [6]:
# ---------------------------------
# 5. Build LangGraph
# ---------------------------------
builder = StateGraph(DecomposeRAGState)

builder.add_node("planner", plan_query)
builder.add_node("retriever", retrieve_for_each)
builder.add_node("responder", generate_final_answer)

builder.set_entry_point("planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "responder")
builder.add_edge("responder", END)

graph = builder.compile()

In [ ]:
# ---------------------------------
# 6. Gradio Interface
# ---------------------------------
def rag_decomposition_pipeline(user_query: str):
    init_state = DecomposeRAGState(question=user_query)
    result = graph.invoke(init_state)
    # Format the sub‑questions into a neat bullet list
    sub_qs = "\n".join([f"- {q}" for q in result["sub_questions"]])
    return (
        f"Sub-questions:\n{sub_qs}\n\n"
        f"Final Answer:\n{result['answer']}"
    )

demo = gr.Interface(
    fn=rag_decomposition_pipeline,
    inputs=gr.Textbox(label="Ask a complex question about the Xumo Manual"),
    outputs=gr.Textbox(label="Decomposed RAG Output"),
    title="Xumo Manual Query Decomposition RAG Assistant"
)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
